In [0]:
%pip install cartopy matplotlib

In [0]:
"""
Plot a bounding box (or polygon) of interest on a map with a lat/lon grid.
"""

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import warnings
warnings.filterwarnings("ignore")

# Define Area of Interest: [min_lon, min_lat, max_lon, max_lat]
BBOX = [-66.0, -27.5, -65, -26.5]  # This can be a polygon too

# How much extra context to show around the box as a fraction of its size
ZOOM_OUT_FACTOR = 2.0

# Minimum padding in degrees (in case the box is very small)
MIN_PADDING_DEG = 2.0

# Country whose provincial/state boundaries should be overlaid (Natural Earth 'admin' field)
PROVINCE_COUNTRY = "Argentina"

def get_extent(bbox, zoom_out_factor=ZOOM_OUT_FACTOR, min_padding=MIN_PADDING_DEG):
    """Compute map extent around the bbox"""
    min_lon, min_lat, max_lon, max_lat = bbox
    width = max_lon - min_lon
    height = max_lat - min_lat

    pad_lon = max(width * zoom_out_factor, min_padding)
    pad_lat = max(height * zoom_out_factor, min_padding)

    return [
        min_lon - pad_lon,
        max_lon + pad_lon,
        min_lat - pad_lat,
        max_lat + pad_lat,
    ]

def add_province_borders(ax, country=PROVINCE_COUNTRY):
    """Overlay admin-1 (state/province) boundaries for a given country."""
    shp_path = shapereader.natural_earth(
        resolution="10m", category="cultural", name="admin_1_states_provinces",
    )
    reader = shapereader.Reader(shp_path)

    for record in reader.records():
        if record.attributes.get("admin") == country:
            ax.add_geometries([record.geometry],crs=ccrs.PlateCarree(),facecolor="none",
                              edgecolor="black", linewidth=0.2, linestyle="-")

def plot_bbox(bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    extent = get_extent(bbox)

    fig = plt.figure(figsize=(9, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    # Base map features
    ax.add_feature(cfeature.LAND, facecolor="#f0efe9")
    ax.add_feature(cfeature.OCEAN, facecolor="#d5e6f2")
    ax.add_feature(cfeature.COASTLINE, linewidth=1)
    ax.add_feature(cfeature.BORDERS, linewidth=1, linestyle="-")
    ax.add_feature(cfeature.LAKES, facecolor="#d5e6f2", edgecolor="black", linewidth=0.3)
    ax.add_feature(cfeature.RIVERS, linewidth=0.3)
    add_province_borders(ax)


    # Lat/lon gridlines with labels
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.6, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False

    # Draw the bounding box
    box_lons = [min_lon, max_lon, max_lon, min_lon, min_lon]
    box_lats = [min_lat, min_lat, max_lat, max_lat, min_lat]
    ax.plot(box_lons, box_lats, color="red", linewidth=1, transform=ccrs.PlateCarree())

    ax.set_title("BBOX area")

    plt.tight_layout()
    plt.show()

    ## If we want to save the map to show:
    # out_path="bbox_map.png"
    # plt.savefig(out_path, dpi=150)
    # print(f"Map saved as: {out_path}")


if __name__ == "__main__":
    plot_bbox(BBOX)